In [ ]:
import rasterio
from rasterio.mask import mask
from shapely.geometry import box
import geopandas as gpd
from google.colab import drive

# 1. Montar Google Drive
drive.mount('/content/drive')

# Define las rutas de tus archivos en el Drive (ajusta la carpeta si es necesario)
ruta_topo = '/content/drive/My Drive/tu_carpeta/30193.tif'
ruta_edaf = '/content/drive/My Drive/tu_carpeta/e1403.tif'
ruta_salida = '/content/drive/My Drive/tu_carpeta/edafologia_recortada_veracruz.tif'

# 2. Obtener la extensión (bounding box) del mapa topográfico
with rasterio.open(ruta_topo) as src_topo:
    topo_bounds = src_topo.bounds
    topo_crs = src_topo.crs
    # Creamos una geometría de caja con los límites del mapa topográfico
    geom = [box(*topo_bounds)]
    print(f"Límites del mapa topográfico: {topo_bounds}")

# 3. Recortar la carta edafológica
with rasterio.open(ruta_edaf) as src_edaf:
    # Es vital que ambos archivos estén en el mismo CRS
    # Si la edafología tiene un CRS distinto, se debe reproyectar la geometría
    # Aquí asumimos que ambos son INEGI y comparten proyección (usualmente UTM 14N o LCC)

    out_image, out_transform = mask(src_edaf, geom, crop=True)
    out_meta = src_edaf.meta.copy()

    # Actualizamos los metadatos para el nuevo archivo recortado
    out_meta.update({
        "driver": "GTiff",
        "height": out_image.shape[1],
        "width": out_image.shape[2],
        "transform": out_transform
    })

# 4. Guardar el resultado en tu Drive
with rasterio.open(ruta_salida, "w", **out_meta) as dest:
    dest.write(out_image)

print(f"Proceso completado. Archivo guardado en: {ruta_salida}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')